# NFSP 德州扑克 AI 训练 (No-Limit Hold'em)

使用 Neural Fictitious Self-Play 在 Google Colab GPU 上训练德州扑克 AI。

**使用方法:**
1. 菜单 → 修改 → 笔记本设置 → 硬件加速器 → 选择 **GPU (T4)**
2. 按顺序运行每个 Cell
3. 训练完成后下载模型文件

**预计时间:**
| 训练量 | T4 GPU | CPU |
|--------|--------|-----|
| 10万局 | ~10分钟 | ~30分钟 |
| 100万局 | ~1.5小时 | ~5小时 |
| 500万局 | ~7小时 | ~25小时 |

## 1. 安装依赖 & 检查 GPU

In [ ]:
!pip install rlcard torch -q

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('⚠️  未检测到GPU! 请在 菜单→修改→笔记本设置 中选择GPU')

## 2. 训练配置

调整以下参数控制训练:

In [ ]:
#@title 训练参数 { display-mode: "form" }

GAME = 'no-limit-holdem'  #@param ['no-limit-holdem', 'limit-holdem', 'leduc-holdem']
NUM_EPISODES = 1000000    #@param {type: "integer"}
EVAL_EVERY = 10000        #@param {type: "integer"}
SAVE_EVERY = 100000       #@param {type: "integer"}
SEED = 42                 #@param {type: "integer"}

# 网络架构
HIDDEN_LAYERS = [512, 512]  # SL (Average Policy) 网络
Q_LAYERS = [512, 512]       # RL (Best Response) 网络

# 超参数
ANTICIPATORY_PARAM = 0.1    # 使用Best Response的概率
RL_LEARNING_RATE = 0.01
SL_LEARNING_RATE = 0.005
BATCH_SIZE = 256
RESERVOIR_SIZE = 2000000

SAVE_DIR = '/content/nfsp_model'

print(f'游戏: {GAME}')
print(f'训练局数: {NUM_EPISODES:,}')
print(f'设备: {"GPU" if torch.cuda.is_available() else "CPU"}')

## 3. 开始训练

In [ ]:
import os
import sys
import io
import time
import json
from datetime import datetime, timedelta
from IPython.display import clear_output
import matplotlib.pyplot as plt

import rlcard
from rlcard.agents import NFSPAgent, RandomAgent
from rlcard.utils import set_seed, reorganize, tournament

def _quiet_feed(agent, ts):
    """Suppress rlcard's print() spam during feed"""
    old_stdout = sys.stdout
    sys.stdout = io.StringIO()
    try:
        agent.feed(ts)
    finally:
        sys.stdout = old_stdout

# Setup
set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs(SAVE_DIR, exist_ok=True)

# Create environments
env = rlcard.make(GAME, config={'seed': SEED})
eval_env = rlcard.make(GAME, config={'seed': SEED + 1})

print(f'环境: {GAME}')
print(f'动作数: {env.num_actions}')
print(f'状态维度: {env.state_shape}')
print(f'设备: {device}')

# Create NFSP agents
agents = []
for i in range(env.num_players):
    agent = NFSPAgent(
        num_actions=env.num_actions,
        state_shape=env.state_shape[i],
        hidden_layers_sizes=HIDDEN_LAYERS,
        q_mlp_layers=Q_LAYERS,
        anticipatory_param=ANTICIPATORY_PARAM,
        rl_learning_rate=RL_LEARNING_RATE,
        sl_learning_rate=SL_LEARNING_RATE,
        reservoir_buffer_capacity=RESERVOIR_SIZE,
        batch_size=BATCH_SIZE,
        device=device,
        evaluate_with='average_policy',
    )
    agents.append(agent)

env.set_agents(agents)
eval_env.set_agents(agents)
print(f'\n✅ 初始化完成: {len(agents)} 个 NFSP agent')

# Training metrics
history = {
    'episodes': [],
    'wr_vs_random': [],
    'wr_vs_self': [],
    'times': [],
}

# Training loop
print(f'\n🚀 开始训练 {NUM_EPISODES:,} 局...\n')
start_time = time.time()

for episode in range(1, NUM_EPISODES + 1):
    # Sample policy
    for agent in agents:
        agent.sample_episode_policy()

    # Run episode
    trajectories, payoffs = env.run(is_training=True)
    trajectories = reorganize(trajectories, payoffs)
    for i, agent in enumerate(agents):
        for ts in trajectories[i]:
            _quiet_feed(agent, ts)

    # Progress display
    if episode % 1000 == 0:
        elapsed = time.time() - start_time
        speed = episode / elapsed
        eta = (NUM_EPISODES - episode) / max(speed, 0.01)
        pct = episode / NUM_EPISODES * 100
        filled = int(30 * episode / NUM_EPISODES)
        bar = '█' * filled + '░' * (30 - filled)
        print(f'\r[{bar}] {pct:5.1f}% | '
              f'{episode:,}/{NUM_EPISODES:,} | '
              f'{speed:.0f} ep/s | '
              f'ETA {timedelta(seconds=int(eta))}',
              end='', flush=True)

    # Evaluation
    if episode % EVAL_EVERY == 0:
        # Evaluate vs Random
        random_agent = RandomAgent(num_actions=eval_env.num_actions)
        eval_env.set_agents([agents[0], random_agent])
        wr_random = tournament(eval_env, 1000)[0]

        # Evaluate self-play
        eval_env.set_agents(agents)
        wr_self = tournament(eval_env, 1000)[0]

        history['episodes'].append(episode)
        history['wr_vs_random'].append(wr_random)
        history['wr_vs_self'].append(wr_self)
        history['times'].append(time.time() - start_time)

        # Restore agents
        env.set_agents(agents)
        eval_env.set_agents(agents)

        # Live plot
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

        ax1.plot(history['episodes'], history['wr_vs_random'], 'b-', label='vs Random')
        ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
        ax1.set_xlabel('Episodes')
        ax1.set_ylabel('Average Payoff')
        ax1.set_title('NFSP vs Random Agent')
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        ax2.plot(history['episodes'], history['wr_vs_self'], 'r-', label='vs Self')
        ax2.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
        ax2.set_xlabel('Episodes')
        ax2.set_ylabel('Average Payoff')
        ax2.set_title('Self-Play Payoff (应趋近0)')
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

        elapsed = time.time() - start_time
        speed = episode / elapsed
        eta = (NUM_EPISODES - episode) / max(speed, 0.01)
        print(f'\n第 {episode:,}/{NUM_EPISODES:,} 局 | '
              f'速度: {speed:.0f} ep/s | '
              f'已用: {timedelta(seconds=int(elapsed))} | '
              f'剩余: {timedelta(seconds=int(eta))}')
        print(f'vs Random: {wr_random:+.4f} | vs Self: {wr_self:+.4f}')

    # Save checkpoint
    if episode % SAVE_EVERY == 0:
        for i, agent in enumerate(agents):
            agent.save_checkpoint(
                path=SAVE_DIR,
                filename=f'agent_{i}_checkpoint.pt'
            )
        print(f'💾 检查点已保存 (第 {episode:,} 局)')

# Final save
total_time = time.time() - start_time
for i, agent in enumerate(agents):
    agent.save_checkpoint(path=SAVE_DIR, filename=f'agent_{i}_checkpoint.pt')

# Final evaluation
random_agent = RandomAgent(num_actions=eval_env.num_actions)
eval_env.set_agents([agents[0], random_agent])
final_wr = tournament(eval_env, 2000)[0]

# Save metrics
with open(os.path.join(SAVE_DIR, 'training_metrics.json'), 'w') as f:
    json.dump({
        'game': GAME,
        'algorithm': 'NFSP',
        'episodes': NUM_EPISODES,
        'total_time': total_time,
        'speed': NUM_EPISODES / total_time,
        'final_wr_vs_random': final_wr,
        'device': str(device),
        'history': history,
    }, f, indent=2)

print(f'\n{"=" * 50}')
print(f'✅ 训练完成!')
print(f'  总局数: {NUM_EPISODES:,}')
print(f'  总耗时: {timedelta(seconds=int(total_time))}')
print(f'  速度:   {NUM_EPISODES/total_time:.0f} ep/s')
print(f'  最终收益 vs Random: {final_wr:+.4f}')
print(f'{"=" * 50}')

## 4. 评估模型

In [ ]:
# 详细评估
print('=' * 50)
print('模型评估报告')
print('=' * 50)

eval_env_final = rlcard.make(GAME, config={'seed': 99})

# vs Random (5000 games)
random_agent = RandomAgent(num_actions=eval_env_final.num_actions)
eval_env_final.set_agents([agents[0], random_agent])
wr = tournament(eval_env_final, 5000)[0]
print(f'\nvs RandomAgent (5000局): {wr:+.4f}')

# Player position analysis
eval_env_final.set_agents([agents[0], random_agent])
wr_p0 = tournament(eval_env_final, 3000)[0]
eval_env_final.set_agents([random_agent, agents[0]])
wr_p1 = tournament(eval_env_final, 3000)[1]
print(f'  位置0 (先手): {wr_p0:+.4f}')
print(f'  位置1 (后手): {wr_p1:+.4f}')

# Self-play (should be close to 0)
eval_env_final.set_agents(agents)
wr_self = tournament(eval_env_final, 3000)
print(f'\n自我对弈 (3000局):')
print(f'  Player 0: {wr_self[0]:+.4f}')
print(f'  Player 1: {wr_self[1]:+.4f}')
print(f'  差值: {abs(wr_self[0] - wr_self[1]):.4f} (越小越好, 接近纳什均衡)')

print(f'\n{"=" * 50}')
if wr > 0.5:
    print('✅ 模型表现优秀 — 大幅领先随机对手')
elif wr > 0.1:
    print('✅ 模型表现良好 — 明显强于随机对手')
elif wr > 0:
    print('⚠️  模型表现一般 — 略强于随机, 建议增加训练量')
else:
    print('❌ 模型需要更多训练 — 建议增加训练局数或调整超参数')

## 5. 下载模型

In [ ]:
# 打包模型文件
import shutil
shutil.make_archive('/content/nfsp_model', 'zip', SAVE_DIR)
print('模型已打包: /content/nfsp_model.zip')
print(f'包含文件:')
for f in os.listdir(SAVE_DIR):
    size = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1024
    print(f'  {f} ({size:.0f} KB)')

# 自动下载 (仅Colab)
try:
    from google.colab import files
    files.download('/content/nfsp_model.zip')
    print('\n📥 下载已开始...')
except:
    print('\n提示: 手动从左侧文件栏下载 nfsp_model.zip')

## 6. 加载模型 (本地使用)

将下载的模型文件放到项目的 `ai_engine/models/nfsp_v1/` 目录下，然后用以下代码加载:

In [ ]:
# 加载示例代码 (在本地项目中使用)
print('''
# === 在你的游戏服务端加载 NFSP 模型 ===

import torch
from rlcard.agents import NFSPAgent

# 加载模型
checkpoint = torch.load('ai_engine/models/nfsp_v1/agent_0_checkpoint.pt',
                        map_location='cpu')
agent = NFSPAgent.from_checkpoint(checkpoint)

# 在游戏中使用
def get_ai_action(game_state):
    """根据当前游戏状态返回AI的动作"""
    action, info = agent.eval_step(game_state)
    # action: 0=fold, 1=check/call, 2=raise_half_pot, 3=raise_pot, 4=all_in
    return action
''')